## Imports

In [1]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning, module="sklearn")
# Suprime os RuntimeWarnings benignos do sklearn (divide by zero / overflow / invalid value
# em matmul) que ocorrem durante o SVD randomizado do PCA em matrizes esparsas de expressão.

from pathlib import Path

import pandas as pd
import numpy as np
import mygene

import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE 
import umap

from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist

/Users/daniel.hosomi/csvd-projects/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configurações

In [2]:
# Diretórios
PROCESSED_DIR = Path("../../data/interim")

# Arquivos de entrada
EXPRESSION_PATH = PROCESSED_DIR / "microplastic_expression.csv"
METADATA_PATH = PROCESSED_DIR / "microplastic_metadata.csv"

# Arquivos de saída
FILTERED_EXPRESSION_PATH = PROCESSED_DIR / "microplastic_expression_filtered.csv"
QC_METRICS_PATH = PROCESSED_DIR / "microplastic_qc_metrics.csv"
LOGCPM_PATH = PROCESSED_DIR / "microplastic_logcpm_filtered.csv"

# Cache da anotação MyGeneInfo — evita chamada à API a cada re-execução
# Para forçar uma atualização da anotação, delete este arquivo e re-execute.
MYGENE_CACHE_PATH = PROCESSED_DIR / "mygene_annotation_cache.csv"

if not EXPRESSION_PATH.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {EXPRESSION_PATH}")

if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {METADATA_PATH}")

## Carregamento e Validação dos Dados Estruturados

In [3]:
expression_df = pd.read_csv(EXPRESSION_PATH)
metadata_df = pd.read_csv(METADATA_PATH)

print("Matriz de expressão:", expression_df.shape)
print("Metadados:", metadata_df.shape)

display(expression_df.head())
display(metadata_df.head())

Matriz de expressão: (63241, 25)
Metadados: (24, 9)


,gene_id,CTR_1,CTR_2,CTR_3,MA1_1,MA1_2,MA1_3,MB1_1,MB1_2,MB1_3,...,MD1_3,MA100_1,MA100_2,MA100_3,MB100_1,MB100_2,MB100_3,MC100_1,MC100_2,MC100_3
0,ENSG00000000003,357,327,329,276,226,355,258,275,464,...,336,336,285,323,333,233,334,310,358,398
1,ENSG00000000005,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,ENSG00000000419,770,453,733,605,629,716,448,748,863,...,969,721,700,733,710,703,633,727,974,712
3,ENSG00000000457,27,115,31,26,33,35,23,32,37,...,47,30,31,7,22,40,44,41,68,67
4,ENSG00000000460,37,27,7,4,5,36,14,45,52,...,0,29,3,0,17,9,19,12,21,33


,particle_type,particle_size_um,particle_size_nm,concentration_gL,is_control,treatment_status,sample_id,group,replicate
0,control,NaN,NaN,0.0,True,control,CTR_1,CTR,1
1,control,NaN,NaN,0.0,True,control,CTR_2,CTR,2
2,control,NaN,NaN,0.0,True,control,CTR_3,CTR,3
3,polystyrene,1.0,1000.0,0.1,False,treated,MA1_1,MA1,1
4,polystyrene,1.0,1000.0,0.1,False,treated,MA1_2,MA1,2


In [4]:
# Verifica consistência entre amostras na matriz de expressão e no metadata

sample_columns = [col for col in expression_df.columns if col != "gene_id"]
metadata_samples = metadata_df["sample_id"].tolist()

missing_in_expression = sorted(set(metadata_samples) - set(sample_columns))
missing_in_metadata = sorted(set(sample_columns) - set(metadata_samples))

print("Amostras na expressão:", len(sample_columns))
print("Amostras no metadata:", len(metadata_samples))

if missing_in_expression:
    print("\nPresentes no metadata, ausentes na expressão:")
    print(missing_in_expression)

if missing_in_metadata:
    print("\nPresentes na expressão, ausentes no metadata:")
    print(missing_in_metadata)

if missing_in_expression or missing_in_metadata:
    raise ValueError("Inconsistência entre matriz de expressão e metadados.")

# Reordena a matriz de expressão conforme a ordem do metadata
expression_df = expression_df[["gene_id"] + metadata_samples]

print("\nOrdem final das amostras alinhada ao metadata.")
print(expression_df.columns.tolist()[1:])

Amostras na expressão: 24
Amostras no metadata: 24

Ordem final das amostras alinhada ao metadata.
['CTR_1', 'CTR_2', 'CTR_3', 'MA1_1', 'MA1_2', 'MA1_3', 'MB1_1', 'MB1_2', 'MB1_3', 'MC1_1', 'MC1_2', 'MC1_3', 'MD1_1', 'MD1_2', 'MD1_3', 'MA100_1', 'MA100_2', 'MA100_3', 'MB100_1', 'MB100_2', 'MB100_3', 'MC100_1', 'MC100_2', 'MC100_3']


# Filtragem do RNAseq para ter somente os que codificam proteína

In [5]:
unique_genes = expression_df["gene_id"].unique().tolist()

if MYGENE_CACHE_PATH.exists():
    # Cache encontrado — carrega localmente (rápido e reprodutível)
    print(f"Carregando anotação do cache: {MYGENE_CACHE_PATH}")
    my_genes_filtereds = pd.read_csv(MYGENE_CACHE_PATH, index_col=0)
else:
    # Primeira execução — consulta a API e salva o resultado em cache
    print("Cache não encontrado — consultando API MyGeneInfo (pode levar ~1 minuto)...")
    mg = mygene.MyGeneInfo()
    my_genes_filtereds = mg.querymany(
        unique_genes,
        scopes='ensembl.gene',
        fields='type_of_gene',
        species='human',
        as_dataframe=True
    )
    my_genes_filtereds.to_csv(MYGENE_CACHE_PATH)
    print(f"Cache salvo em: {MYGENE_CACHE_PATH}")

protein_genes = my_genes_filtereds[my_genes_filtereds['type_of_gene'] == 'protein-coding'].index.tolist()
print(f"Identificados {len(protein_genes)} genes codificadores de proteínas.")
expression_df_filtered = expression_df[expression_df['gene_id'].isin(protein_genes)].copy()

Carregando anotação do cache: ../../data/interim/mygene_annotation_cache.csv
Identificados 19519 genes codificadores de proteínas.


### Genes removidos pela anotação MyGeneInfo

A consulta `mg.querymany()` mapeia os IDs Ensembl para metadados de genes humanos. Três categorias de genes são descartadas:

| Categoria | Quantidade | Motivo |
|---|---|---|
| **Hits duplicados** | 37 genes | Mapeamentos ambíguos no Ensembl (um ID mapeia para mais de um gene) |
| **Sem correspondência** | 1.241 genes | Provavelmente pseudogenes, genes não catalogados ou IDs desatualizados na versão do Ensembl utilizada |
| **Não codificadores de proteína** | ~42.449 genes | lncRNA, miRNA, snoRNA, rRNA, etc. — excluídos da análise funcional |

**Resultado:** de 63.241 genes totais, **19.515 são codificadores de proteína** (~30,9%) e entram nas etapas seguintes.

> Essa filtragem é esperada e garante que apenas genes funcionalmente relevantes para a pergunta de pesquisa sejam analisados.

In [6]:
counts_df = expression_df_filtered.set_index("gene_id")

print("Dimensões da matriz de contagens:", counts_df.shape)

# Valores ausentes
n_missing = counts_df.isna().sum().sum()
print("Número total de valores ausentes:", n_missing)

# Tipos
print("\nTipos das colunas:")
print(counts_df.dtypes.value_counts())

# Verifica contagens negativas
has_negative = (counts_df < 0).any().any()
print("\nExistem contagens negativas?", has_negative)

# Verifica se todos os valores parecem inteiros
is_integer_like = np.all(np.equal(np.mod(counts_df.fillna(0).to_numpy(), 1), 0))
print("Todos os valores são inteiros?", is_integer_like)

Dimensões da matriz de contagens: (19515, 24)
Número total de valores ausentes: 0

Tipos das colunas:
int64    24
Name: count, dtype: int64

Existem contagens negativas? False
Todos os valores são inteiros? True


## Calcula Métricas para Controle de Qualidade

In [7]:
# Calcula métricas de qualidade para cada amostra
# - library_size: soma total de contagens por amostra
# - detected_genes_gt0: número de genes com contagem > 0
# - p1 a p99: percentis da distribuição de contagens dos genes em cada amostra

qc_metrics = pd.DataFrame({
    "sample_id": counts_df.columns,
    "library_size": counts_df.sum(axis=0).values,
    "detected_genes_gt0": (counts_df > 0).sum(axis=0).values,
    
    # Cálculo dos percentis utilizando o método .quantile()
    "p1": counts_df.quantile(0.01).values,
    "p50": counts_df.quantile(0.50).values,
    "p75": counts_df.quantile(0.75).values,
    "p83": counts_df.quantile(0.83).values,
    "p90": counts_df.quantile(0.90).values,
    "p95": counts_df.quantile(0.95).values,
    "p99": counts_df.quantile(0.99).values,
})

# Junta as métricas calculadas com os metadados existentes
qc_metrics = qc_metrics.merge(metadata_df, on="sample_id", how="left")

# Mostra o resultado final ordenado por grupo e réplica
display(qc_metrics.sort_values(["group", "replicate"]))

,sample_id,library_size,detected_genes_gt0,p1,p50,p75,p83,p90,p95,p99,particle_type,particle_size_um,particle_size_nm,concentration_gL,is_control,treatment_status,group,replicate
0,CTR_1,12294780,12930,0.0,40.0,393.0,676.00,1196.6,2221.3,9003.56,control,NaN,NaN,0.000,True,control,CTR,1
1,CTR_2,11753669,12496,0.0,38.0,373.0,646.00,1149.0,2138.5,8808.72,control,NaN,NaN,0.000,True,control,CTR,2
2,CTR_3,11119923,12617,0.0,34.0,364.0,617.00,1115.6,2021.0,8076.72,control,NaN,NaN,0.000,True,control,CTR,3
3,MA1_1,10497166,12760,0.0,33.0,339.0,574.00,1032.0,1916.9,7648.48,polystyrene,1.0,1000.0,0.100,False,treated,MA1,1
4,MA1_2,11003511,12484,0.0,36.0,356.0,603.00,1097.2,2023.0,8021.16,polystyrene,1.0,1000.0,0.100,False,treated,MA1,2
5,MA1_3,13627165,13760,0.0,51.0,477.0,807.00,1422.2,2604.6,9615.36,polystyrene,1.0,1000.0,0.100,False,treated,MA1,3
15,MA100_1,12451358,12167,0.0,43.0,421.0,734.00,1317.6,2412.0,9214.72,polystyrene,0.1,100.0,0.100,False,treated,MA100,1
16,MA100_2,10569761,12347,0.0,34.0,348.0,601.00,1083.0,2001.3,7817.98,polystyrene,0.1,100.0,0.100,False,treated,MA100,2
17,MA100_3,13910517,12339,0.0,48.0,481.0,802.62,1466.6,2681.4,10167.86,polystyrene,0.1,100.0,0.100,False,treated,MA100,3
6,MB1_1,8690949,12451,0.0,29.0,287.0,486.00,883.6,1644.0,6603.62,polystyrene,1.0,1000.0,0.010,False,treated,MB1,1


In [8]:
fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=(
        "Library size por amostra", "Genes detectados (> 0)",
        "Percentil 75", "Percentil 83",
        "Percentil 90", "Percentil 95", "Percentil 99"
    )
)

metricas = [
    ("library_size", "Soma das contagens", 1, 1),
    ("detected_genes_gt0", "Número de genes", 1, 2),
    ("p75", "Contagem de genes (p75)", 1, 3),
    ("p83", "Contagem de genes (p83)", 1, 4),
    ("p90", "Contagem de genes (p90)", 2, 1),
    ("p95", "Contagem de genes (p95)", 2, 2),
    ("p99", "Contagem de genes (p99)", 2, 3)
]

for coluna, eixo_y, linha, col in metricas:
    fig.add_trace(
        go.Bar(x=qc_metrics["sample_id"], y=qc_metrics[coluna], showlegend=False),
        row=linha, col=col
    )
    fig.update_yaxes(title_text=eixo_y, row=linha, col=col)

fig.update_layout(height=1200, width=1400)
fig.update_xaxes(tickangle=90)

fig.show()

## Funções Auxiliares para CPM (Count-per-Million) e Visualização

In [9]:
def compute_cpm(counts: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula counts-per-million (CPM) por amostra, normalizando pela library size.
    """
    library_sizes = counts.sum(axis=0)
    cpm = counts.divide(library_sizes, axis=1) * 1_000_000
    return cpm


def compute_logcpm(counts: pd.DataFrame, prior_count: float = 1.0) -> pd.DataFrame:
    """
    Calcula log2(CPM + prior_count), útil para PCA e inspeção visual devido à assimetria de CPM.
    """
    cpm = compute_cpm(counts)
    return np.log2(cpm + prior_count)

def plot_dim_reduction(counts_df: pd.DataFrame, metadata_df: pd.DataFrame, metodo: str = "PCA") -> go.Figure:
    logcpm_df = compute_logcpm(counts_df)
    X = logcpm_df.T.values
    
    metodo_limpo = metodo.strip().upper()
    
    if metodo_limpo == "PCA":
        modelo = PCA(n_components=2, random_state=42)
        resultados = modelo.fit_transform(X)
        eixo_x, eixo_y = "PC1", "PC2"
        titulo = "PCA das amostras (logCPM)"
        
    elif metodo_limpo == "TSNE":
        n_amostras = X.shape[0]
        tsne_perplexity = min(30, n_amostras - 1)
        modelo = TSNE(n_components=2, perplexity=tsne_perplexity, random_state=42)
        resultados = modelo.fit_transform(X)
        eixo_x, eixo_y = "tSNE1", "tSNE2"
        titulo = "t-SNE das amostras (logCPM)"
        
    elif metodo_limpo == "UMAP":
        modelo = umap.UMAP(n_components=2, random_state=42)
        resultados = modelo.fit_transform(X)
        eixo_x, eixo_y = "UMAP1", "UMAP2"
        titulo = "UMAP das amostras (logCPM)"
        
    else:
        raise ValueError("Método inválido! Escolha 'PCA', 'TSNE' ou 'UMAP'.")

    dimred_df = pd.DataFrame({
        "sample_id": logcpm_df.columns,
        eixo_x: resultados[:, 0],
        eixo_y: resultados[:, 1],
    })
    
    dimred_df = dimred_df.merge(metadata_df, on="sample_id", how="left")
    
    fig = px.scatter(
        dimred_df, x=eixo_x, y=eixo_y, color="group", text="sample_id",
        title=titulo
    )
    
    fig.update_traces(textposition="top center", marker=dict(size=12))
    fig.update_layout(height=600, width=800)
    
    return fig

## Exploração dos Dados

In [10]:
# Visualizações de PCA, t-SNE e UMAP dos dados em 2 dimensões usando logCPM
pca_fig = plot_dim_reduction(counts_df, metadata_df, metodo="PCA")
pca_fig.show()

tsne_fig = plot_dim_reduction(counts_df, metadata_df, metodo="TSNE")
tsne_fig.show()

umap_fig = plot_dim_reduction(counts_df, metadata_df, metodo="UMAP")
umap_fig.show()

/Users/daniel.hosomi/csvd-projects/.venv/lib/python3.9/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [11]:
logcpm_df = compute_logcpm(counts_df)

fig = ff.create_dendrogram(
    logcpm_df.T.values,
    labels=logcpm_df.columns.tolist(),
    linkagefun=lambda x: linkage(x, method="average", metric="euclidean")
)

fig.update_layout(
    title_text="Clustering hierárquico das amostras",
    yaxis_title="Distância",
    width=1000,
    height=500
)
fig.update_xaxes(tickangle=90)
fig.show()

## Filtragem de Genes Pouco Expressos

Dois critérios complementares são aplicados em conjunto:

| Critério | Valor | Justificativa |
|---|---|---|
| **CPM ≥ 1 em ≥ 3 amostras** | `MIN_CPM=1.0`, `MIN_SAMPLES=3` | Garante expressão detectável em pelo menos 1 grupo biológico completo (cada grupo tem 3 réplicas). Threshold padrão da literatura para RNA-seq. |
| **Contagem total ≥ 100** | `MIN_TOTAL_COUNT=100` | Remove genes com expressão muito baixa mesmo passando pelo critério de CPM (ex: CPM ≥ 1 em exatamente 3 amostras de biblioteca pequena). Aumenta a robustez das estimativas de correlação no WGCNA. |

> **Nota:** o filtro de CPM já é bastante restritivo — qualquer gene com CPM ≥ 1 em ≥ 3 amostras de biblioteca ~8,7M tem contagem total ≥ 26. O limiar de 100 atua como filtro adicional para genes marginais e é o valor com o qual todos os artefatos downstream (DESeq2, WGCNA) foram gerados.

In [12]:
# Critérios de filtragem

MIN_CPM = 1.0
MIN_SAMPLES = 3
MIN_TOTAL_COUNT = 100

cpm_df = compute_cpm(counts_df)

keep_by_cpm = (cpm_df >= MIN_CPM).sum(axis=1) >= MIN_SAMPLES
keep_by_total_count = counts_df.sum(axis=1) >= MIN_TOTAL_COUNT

keep_mask = keep_by_cpm & keep_by_total_count

filtered_counts_df = counts_df.loc[keep_mask].copy()

print("Genes antes da filtragem:", counts_df.shape[0])
print("Genes após a filtragem:", filtered_counts_df.shape[0])
print(f"Genes removidos: {counts_df.shape[0] - filtered_counts_df.shape[0]}")

Genes antes da filtragem: 19515
Genes após a filtragem: 12174
Genes removidos: 7341


In [13]:
# Visualizações de PCA, t-SNE e UMAP dos dados em 2 dimensões usando logCPM após filtragem dos dados
pca_fig = plot_dim_reduction(filtered_counts_df, metadata_df, metodo="PCA")
pca_fig.show()

tsne_fig = plot_dim_reduction(filtered_counts_df, metadata_df, metodo="TSNE")
tsne_fig.show()

umap_fig = plot_dim_reduction(filtered_counts_df, metadata_df, metodo="UMAP")
umap_fig.show()

/Users/daniel.hosomi/csvd-projects/.venv/lib/python3.9/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## Salvamento dos Artefatos

In [14]:
filtered_logcpm_df = compute_logcpm(filtered_counts_df)
filtered_expression_df = filtered_counts_df.reset_index()

filtered_expression_df.to_csv(FILTERED_EXPRESSION_PATH, index=False)
qc_metrics.to_csv(QC_METRICS_PATH, index=False)
filtered_logcpm_df.reset_index().to_csv(LOGCPM_PATH, index=False)

print("Arquivos salvos:")
print("-", FILTERED_EXPRESSION_PATH)
print("-", QC_METRICS_PATH)
print("-", LOGCPM_PATH)

Arquivos salvos:
- ../../data/interim/microplastic_expression_filtered.csv
- ../../data/interim/microplastic_qc_metrics.csv
- ../../data/interim/microplastic_logcpm_filtered.csv
